# 📊 SPSS → Makale Üretici

SPSS **.sav** dosyanızdan **Türkçe akademik makale taslağı (Word)** üretir.
Ücretsiz, kurulumsuz, sunucusuz — iPhone Safari'de bile çalışır.

**Kullanım:** Üstten **Çalışma Zamanı → Tümünü Çalıştır** (Runtime → Run all). Sonra `.sav` dosyanızı seçin; biraz bekleyin; Word otomatik inecek.

Sayılar istatistik motorundan **birebir** gelir (uydurma yok); kimlik/PII değişkenleri otomatik dışlanır ve veriniz yalnızca bu oturumda kalır.


In [ ]:
#@title 1) Gerekli kütüphaneleri kur (1-2 dk)
!pip -q install pyreadstat pingouin scikit-posthocs factor_analyzer python-docx jsonschema tabulate httpx 2>/dev/null
print('Kurulum tamam.')


In [ ]:
#@title 2) Makale motorunu yükle (pakete gömülü)
import base64, io, tarfile, sys
_PKG = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(_PKG)), mode='r:gz').extractall('/content')
sys.path.insert(0, '/content')
import sav2q1; print('Motor yüklendi, sürüm', sav2q1.__version__)


In [ ]:
#@title 3) SPSS .sav yükle → Makale üret → Word indir
with_pubmed = False  #@param {type:'boolean'}
konu = ''  #@param {type:'string'}
from google.colab import files
from sav2q1.pipeline import generate_article
print('SPSS .sav dosyanızı seçin…')
up = files.upload()
sav_name = list(up.keys())[0]
brief = {'topic': konu} if konu.strip() else None
res = generate_article(sav_name, '/content/run', brief=brief, with_pubmed=with_pubmed)
print('\nGruplama değişkeni:', res['group_var'], '| Sonuç:', res['n_results'],
      '| Sayı doğrulama:', res['gate'].get('numeric'))
print('Word indiriliyor…')
files.download(res['docx'])


---
**Notlar:** Etik kurul no, finansman ve yazar alanları Word'de `[köşeli parantez]` ile işaretlidir; siz doldurun. **Gerçek PubMed kaynakları** için 3. hücrede `with_pubmed`'i işaretleyip bir **konu** yazın (ör. *nonalcoholic fatty liver children*).
